# 04 — AI ingredient matching vs RapidFuzz baseline

**Riskiest question:** can we match each recipe ingredient to the right Agribalyse food?

**Method (fair comparison):**
1. Same 100 hand-verified labels for both methods (`data/ingredient_labels.csv`).
2. Model and "uncertain" threshold are chosen on the **tune** split (50 rows) only.
3. Accuracy is reported once on the **test** split (50 rows), never used to choose anything.

**Decision rule:** keep the AI matcher only if it gets **at least 10 percentage points more** than RapidFuzz on test (≥ 5 more correct out of 50).

> Run with `Runtime → Run all`. A GPU is optional (`Runtime → Change runtime type → T4 GPU` makes it faster).

## 1. Get the code and data

In [ ]:
# If the repo is private, use: !git clone https://<YOUR_TOKEN>@github.com/laurarigau/Food-Sustainability.git
!git clone https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt -r requirements-ai.txt

In [ ]:
import sys
sys.path.insert(0, "src")
import pandas as pd
from evaluate_matching import build_matcher, evaluate, load_labels
from ai_matching import CANDIDATE_MODELS, EmbeddingMatcher
import numpy as np

labels = load_labels()
print(labels.split.value_counts())

## 2. Baseline: RapidFuzz

In [ ]:
rf_matcher, rf_grid = build_matcher("rapidfuzz")
rf = evaluate(rf_matcher, rf_grid, labels)
print(f"RapidFuzz | threshold (tune) {rf['threshold']} | tune {rf['tune_acc']:.0%} | TEST {rf['test_correct']}/{rf['test_n']}")

## 3. Choose the AI model on the TUNE split
We compare three multilingual models. Only tune accuracy is used to choose; test is not looked at here.

In [ ]:
ai_grid = np.round(np.arange(0, 1.0001, 0.05), 2)
results = {}
for name in CANDIDATE_MODELS:
    m = EmbeddingMatcher(model_name=name, threshold=0.0)
    r = evaluate(m, ai_grid, labels)
    results[name] = (m, r)
    print(f"{name:65s} | threshold {r['threshold']:.2f} | tune {r['tune_acc']:.0%}")

best_name = max(results, key=lambda n: results[n][1]["tune_acc"])
best_matcher, best = results[best_name]
print("\nChosen on tune:", best_name, "| threshold", best["threshold"])

## 4. Final result on TEST

In [ ]:
ai_correct, rf_correct, n = best["test_correct"], rf["test_correct"], best["test_n"]
print(f"RapidFuzz : {rf_correct}/{n} ({rf_correct/n:.0%})")
print(f"AI        : {ai_correct}/{n} ({ai_correct/n:.0%})  [{best_name}]")
diff_points = (ai_correct - rf_correct) / n * 100
print(f"\nDifference: {diff_points:+.0f} percentage points")
print("DECISION:", "CONTINUE with the AI matcher" if diff_points >= 10 else "STOP: use RapidFuzz and declare it")

by_lang = pd.DataFrame({
    "rapidfuzz": rf["test"].groupby("language")["correct"].sum(),
    "ai": best["test"].groupby("language")["correct"].sum(),
    "n": best["test"].groupby("language")["correct"].count(),
})
by_lang

## 5. Row by row: where each method wins or fails

In [ ]:
cmp = best["test"][["label_id", "ingredient_text", "language", "agribalyse_name"]].copy()
cmp["rapidfuzz_pred"] = rf["test"]["pred_name"].values
cmp["rapidfuzz_ok"] = rf["test"]["correct"].values
cmp["ai_pred"] = np.where(best["test"]["uncertain"], "UNCERTAIN", best["test"]["pred_name"])
cmp["ai_score"] = best["test"]["score"].round(2).values
cmp["ai_ok"] = best["test"]["correct"].values
pd.set_option("display.max_colwidth", 45)
cmp

In [ ]:
print("AI right, RapidFuzz wrong:", ((cmp.ai_ok) & (~cmp.rapidfuzz_ok)).sum())
print("RapidFuzz right, AI wrong:", ((~cmp.ai_ok) & (cmp.rapidfuzz_ok)).sum())
cmp[~cmp.ai_ok]  # remaining AI errors: discuss them in the log

## 6. Save results for the repo

In [ ]:
best["test"].to_csv("data/results_ai_test.csv", index=False)
cmp.to_csv("data/comparison_ai_vs_rapidfuzz_test.csv", index=False)
summary = pd.DataFrame([
    {"matcher": "rapidfuzz", "model": "WRatio", "threshold": rf["threshold"], "tune_acc": rf["tune_acc"], "test_correct": rf_correct, "test_n": n},
    *[{"matcher": "ai", "model": k, "threshold": v[1]["threshold"], "tune_acc": v[1]["tune_acc"],
       "test_correct": v[1]["test_correct"] if k == best_name else None, "test_n": n} for k, v in results.items()],
])
summary.to_csv("data/matching_summary.csv", index=False)
summary

In [ ]:
from google.colab import files
for f in ["data/results_ai_test.csv", "data/comparison_ai_vs_rapidfuzz_test.csv", "data/matching_summary.csv"]:
    files.download(f)

## 7. Next
- Put the chosen model name and threshold in `src/ai_matching.py` (`DEFAULT_MODEL`, `DEFAULT_THRESHOLD`).
- Save this notebook **with outputs** to `notebooks/04_ai_matching.ipynb` (`File → Download → .ipynb`) and commit it with the 3 CSV files.
- Write log entry 06 with the numbers above and the decision.